# Auto Schema Generation for Tools

In [ ]:
%run shared_notebook_setup.py

We will auto-generate the OpenAI tool schema directly from this function's type hints and docstring.

In [ ]:
def get_weather(location: str, unit: str = "celsius"):
    """Get current weather for a city.

    Args:
        location: City name, e.g. Seattle
        unit: celsius or fahrenheit
    """
    sample = {
        "seattle": {"celsius": 16, "fahrenheit": 61, "condition": "Cloudy"},
        "austin": {"celsius": 31, "fahrenheit": 88, "condition": "Hot"},
        "new york": {"celsius": 22, "fahrenheit": 72, "condition": "Sunny"},
    }
    weather = sample.get(location.strip().lower(), {"celsius": 24, "fahrenheit": 75, "condition": "Clear"})
    return {
        "location": location,
        "unit": unit,
        "temperature": weather[unit],
        "condition": weather["condition"],
    }

## 2) Auto-Build the Tool Schema from Signature

This helper inspects function parameters and creates `tools=[...]` automatically.

In [ ]:
import inspect
import json
from typing import get_origin, get_args

# this function converts Python type annotations to JSON Schema types, 
# which is used to define the parameters for the tool in a way that the LLM can understand. 
# It handles basic types like str, int, float, bool, as well as lists and dictionaries.
def _python_type_to_json_schema(annotation):
    if annotation is inspect._empty:
        return {"type": "string"}

    if annotation is str:
        return {"type": "string"}
    if annotation is int:
        return {"type": "integer"}
    if annotation is float:
        return {"type": "number"}
    if annotation is bool:
        return {"type": "boolean"}

    # get_origin (from Python’s typing) tells you the “base” type behind a type hint.
    # get_origin(List[int]) --> list, 
    origin = get_origin(annotation)
    args = get_args(annotation)

    if origin is list and args:
        return {"type": "array", "items": _python_type_to_json_schema(args[0])}

    if origin is dict:
        return {"type": "object"}

    return {"type": "string"}

def build_tool_from_function(func, enums=None):
    
    # This function takes a Python function and builds a tool definition that includes 
    # the function's name, description, and parameters based on its type annotations.
    # Pass an 'enums' dict of parameter name -> allowed values to restrict a parameter's values.
    sig = inspect.signature(func)
    properties = {}
    required = []

    for name, param in sig.parameters.items():
        schema = _python_type_to_json_schema(param.annotation)

        if enums and name in enums:
            schema["enum"] = enums[name]

        properties[name] = schema

        if param.default is inspect._empty:
            required.append(name)

    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": (func.__doc__ or "").strip() or f"Call {func.__name__}",
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }


# Restrict the 'unit' parameter to these allowed values via the enums argument
weather_tool = build_tool_from_function(get_weather, enums={"unit": ["celsius", "fahrenheit"]})
tools = [weather_tool]

print(json.dumps(weather_tool, indent=2))

## 3) Use the Auto-Generated Tool in a Real Call

In [ ]:
messages = [
    {"role": "system", "content": "You are helpful. Use tools when needed."},
    {"role": "user", "content": "What is the weather in Seattle in fahrenheit?"},
]

first = client.chat.completions.create(
    model=DATABRICKS_ENDPOINT,
    messages=messages,
    tools=tools,
    tool_choice="auto",
    temperature=0,
)

assistant_msg = first.choices[0].message
messages.append(
    {
        "role": "assistant",
        "content": assistant_msg.content or "",
        "tool_calls": [tc.model_dump() for tc in (assistant_msg.tool_calls or [])],
    }
)

if assistant_msg.tool_calls:
    tc = assistant_msg.tool_calls[0]
    args = json.loads(tc.function.arguments)

    tool_result = get_weather(**args)

    messages.append(
        {
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(tool_result),
        }
    )

    final = client.chat.completions.create(
        model=DATABRICKS_ENDPOINT,
        messages=messages,
        tools=tools,
        temperature=0,
    )

    print("Tool picked by model:", tc.function.name)
    print("Tool args:", args)
    print("Tool result:", tool_result)
    print("Final answer:", final.choices[0].message.content)
else:
    print("No tool call returned. Re-run this cell.")

## 4) Reuse the Same Builder for Another Function

Here we define `add_numbers` and generate its schema with the same `build_tool_from_function(...)` helper.

In [ ]:
def add_numbers(a: int, b: int):
    """Add two integers and return the sum."""
    return {"sum": a + b}


add_tool = build_tool_from_function(add_numbers)
print("Auto-generated schema for add_numbers:\n")
print(json.dumps(add_tool, indent=2))

messages_add = [
    {"role": "user", "content": "What is 17 + 25? Use the add_numbers tool."},
]

first_add = client.chat.completions.create(
    model=DATABRICKS_ENDPOINT,
    messages=messages_add,
    tools=[add_tool],
    tool_choice="auto",
    temperature=0,
)

assistant_add = first_add.choices[0].message
messages_add.append(
    {
        "role": "assistant",
        "content": assistant_add.content or "",
        "tool_calls": [tc.model_dump() for tc in (assistant_add.tool_calls or [])],
    }
)

if assistant_add.tool_calls:
    tc = assistant_add.tool_calls[0]
    args = json.loads(tc.function.arguments)
    tool_result = add_numbers(**args)

    messages_add.append(
        {
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(tool_result),
        }
    )

    final_add = client.chat.completions.create(
        model=DATABRICKS_ENDPOINT,
        messages=messages_add,
        tools=[add_tool],
        temperature=0,
    )

    print("\nTool picked:", tc.function.name)
    print("Tool args:", args)
    print("Tool result:", tool_result)
    print("Final answer:", final_add.choices[0].message.content)
else:
    print("No tool call returned. Re-run this cell.")